<a href="https://colab.research.google.com/github/Rohil121/bharat-portfolio-lab/blob/v0.3-robustness-testing/notebooks/03_robustness_testing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bharat Portfolio Lab — Robustness Testing

## Version v0.3

This notebook evaluates whether the Adaptive Barbell strategy remains reliable when the testing period, parameters and modelling assumptions change.

### Planned tests

- Development-period versus out-of-sample performance
- Momentum, volatility and trend-window sensitivity
- Transaction-cost stress testing
- Rolling return and risk analysis
- Market-regime performance
- Strategy robustness scoring

The objective is not to find the historically best parameter combination. The objective is to identify whether reasonable alternative assumptions produce broadly consistent results.

## 1. Environment Setup

This section prepares the Python environment for reproducible robustness testing of the Adaptive Barbell strategy.

In [1]:
%pip install -q yfinance

In [2]:
import sys
from importlib.metadata import version
from itertools import product

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yfinance as yf

# Reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Display settings
pd.set_option("display.max_columns", 80)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

print("v0.3 environment setup complete.")
print("-" * 45)
print(f"Python: {sys.version.split()[0]}")
print(f"pandas: {pd.__version__}")
print(f"NumPy: {np.__version__}")
print(f"Matplotlib: {version('matplotlib')}")
print(f"yfinance: {version('yfinance')}")
print(f"Random seed: {RANDOM_SEED}")

v0.3 environment setup complete.
---------------------------------------------
Python: 3.12.13
pandas: 2.2.2
NumPy: 2.0.2
Matplotlib: 3.10.0
yfinance: 0.2.66
Random seed: 42


## 2. Portfolio and Robustness-Test Configuration

The v0.3 analysis retains the same India 10 flagship portfolio so that the strategy remains comparable with version v0.2.

The historical period is divided into:

- **Development period:** used to examine and establish the strategy rules.
- **Out-of-sample period:** used to evaluate the frozen strategy on later data.

The out-of-sample results will receive greater importance than the full-period results.

In [3]:
# ---------------------------------------------------------
# India 10 Adaptive Barbell portfolio
# ---------------------------------------------------------

portfolio = pd.DataFrame(
    {
        "Company": [
            "HDFC Bank",
            "Tata Consultancy Services",
            "Hindustan Unilever",
            "Sun Pharmaceutical Industries",
            "Power Grid Corporation of India",
            "Bharti Airtel",
            "Larsen & Toubro",
            "Mahindra & Mahindra",
            "Bharat Electronics",
            "Trent",
        ],
        "Ticker": [
            "HDFCBANK.NS",
            "TCS.NS",
            "HINDUNILVR.NS",
            "SUNPHARMA.NS",
            "POWERGRID.NS",
            "BHARTIARTL.NS",
            "LT.NS",
            "M&M.NS",
            "BEL.NS",
            "TRENT.NS",
        ],
        "Sector": [
            "Financial Services",
            "Information Technology",
            "Consumer Staples",
            "Healthcare",
            "Utilities",
            "Telecommunication",
            "Industrials",
            "Automobile",
            "Defence Electronics",
            "Consumer Retail",
        ],
        "Strategy Bucket": [
            "Resilient Compounder",
            "Resilient Compounder",
            "Resilient Compounder",
            "Resilient Compounder",
            "Resilient Compounder",
            "Growth Leader",
            "Growth Leader",
            "Growth Leader",
            "Growth Leader",
            "Growth Leader",
        ],
    }
)

portfolio["Initial Weight"] = 1 / len(portfolio)

portfolio_tickers = portfolio["Ticker"].tolist()

resilient_tickers = portfolio.loc[
    portfolio["Strategy Bucket"] == "Resilient Compounder",
    "Ticker",
].tolist()

growth_tickers = portfolio.loc[
    portfolio["Strategy Bucket"] == "Growth Leader",
    "Ticker",
].tolist()


# ---------------------------------------------------------
# Market and portfolio assumptions
# ---------------------------------------------------------

BENCHMARK_TICKER = "^NSEI"
BENCHMARK_NAME = "Nifty 50"

INITIAL_CAPITAL = 1_000_000  # ₹10 lakh
TRADING_DAYS = 252
RISK_FREE_RATE = 0.065

TRANSACTION_COST_RATE = 0.0015


# ---------------------------------------------------------
# Historical testing periods
# ---------------------------------------------------------

DATA_START_DATE = "2016-01-01"

DEVELOPMENT_START_DATE = pd.Timestamp("2016-01-01")
OUT_OF_SAMPLE_START_DATE = pd.Timestamp("2022-01-01")

DOWNLOAD_END_DATE = (
    pd.Timestamp.today().normalize()
    + pd.Timedelta(days=1)
)


# ---------------------------------------------------------
# Frozen v0.2 strategy parameters
# ---------------------------------------------------------

BASE_MOMENTUM_LOOKBACK = 126
BASE_VOLATILITY_LOOKBACK = 63
BASE_TREND_LOOKBACK = 200

BASE_FULL_EQUITY_EXPOSURE = 1.00
BASE_DEFENSIVE_EQUITY_EXPOSURE = 0.70


# ---------------------------------------------------------
# Validation checks
# ---------------------------------------------------------

assert len(portfolio) == 10
assert portfolio["Ticker"].is_unique
assert len(resilient_tickers) == 5
assert len(growth_tickers) == 5
assert np.isclose(portfolio["Initial Weight"].sum(), 1.0)

assert DEVELOPMENT_START_DATE < OUT_OF_SAMPLE_START_DATE
assert 0 <= BASE_DEFENSIVE_EQUITY_EXPOSURE <= 1
assert BASE_FULL_EQUITY_EXPOSURE == 1.0


print("v0.3 configuration completed.")
print("-" * 55)
print(f"Portfolio holdings: {len(portfolio)}")
print(f"Data begins: {DATA_START_DATE}")
print(
    "Development period:",
    f"{DEVELOPMENT_START_DATE.date()} to "
    f"{(OUT_OF_SAMPLE_START_DATE - pd.Timedelta(days=1)).date()}",
)
print(
    "Out-of-sample period:",
    f"{OUT_OF_SAMPLE_START_DATE.date()} onward",
)
print(f"Initial capital: ₹{INITIAL_CAPITAL:,.0f}")
print(f"Base transaction cost: {TRANSACTION_COST_RATE:.2%}")

display(portfolio)

v0.3 configuration completed.
-------------------------------------------------------
Portfolio holdings: 10
Data begins: 2016-01-01
Development period: 2016-01-01 to 2021-12-31
Out-of-sample period: 2022-01-01 onward
Initial capital: ₹1,000,000
Base transaction cost: 0.15%


,Company,Ticker,Sector,Strategy Bucket,Initial Weight
0,HDFC Bank,HDFCBANK.NS,Financial Services,Resilient Compounder,0.1000
1,Tata Consultancy Services,TCS.NS,Information Technology,Resilient Compounder,0.1000
2,Hindustan Unilever,HINDUNILVR.NS,Consumer Staples,Resilient Compounder,0.1000
3,Sun Pharmaceutical Industries,SUNPHARMA.NS,Healthcare,Resilient Compounder,0.1000
4,Power Grid Corporation of India,POWERGRID.NS,Utilities,Resilient Compounder,0.1000
5,Bharti Airtel,BHARTIARTL.NS,Telecommunication,Growth Leader,0.1000
6,Larsen & Toubro,LT.NS,Industrials,Growth Leader,0.1000
7,Mahindra & Mahindra,M&M.NS,Automobile,Growth Leader,0.1000
8,Bharat Electronics,BEL.NS,Defence Electronics,Growth Leader,0.1000
9,Trent,TRENT.NS,Consumer Retail,Growth Leader,0.1000
